<a href="https://colab.research.google.com/github/RafaelKobayashiSantos/Google-Maps-Scrapper/blob/main/ScrapMaps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Para rodar tudo de uma vez: (Recomendado)**

  **Clique em "Ambiente de execução" -> "Executar tudo"**

  Feito isso, o código irá te redirecionar perguntando o que quer pesquisar no google maps. Terminando a extração, o resultado será baixado em forma de planilha.

# **OU rode uma célula por vez.**

  No final, terá uma célula onde vai solicitar uma query, e a seguinte basta rodar para baixar o arquivo.


# 1 - Instaladores
  Necessário rodar, leva algum tempinho

In [1]:
# Instala as dependências do sistema necessárias para o Chromium no Colab
!apt-get update
!apt-get install -y \
    libatk1.0-0 \
    libatk-bridge2.0-0 \
    libcups2 \
    libxkbcommon0 \
    libxcomposite1 \
    libxdamage1 \
    libxfixes3 \
    libxrandr2 \
    libgbm1 \
    libasound2 \
    libpango-1.0-0 \
    libpangocairo-1.0-0

# Instala o playwright e os browsers
!pip install playwright nest_asyncio
!playwright install chromium

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,228 kB]
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,307 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,030 kB

# 2- Código
Lógica criada para a extração, pode levar
alguns minutos para terminar a extração.


In [2]:
import asyncio
from playwright.async_api import async_playwright
import time
import re
import nest_asyncio
nest_asyncio.apply()

# ------------------------
# LIMPEZA
# ------------------------
def limpar_texto(texto):
    texto = re.sub(r'[\ue000-\uf8ff]', '', texto)
    return texto.strip()

# ------------------------
# EXTRAÇÃO DE DETALHES
# ------------------------
async def extrair_detalhes(page, card):
    try:
        await card.click()
        await page.wait_for_selector("div[role='main']", timeout=5000)
        await asyncio.sleep(0.5)

        telefone = "Não encontrado"
        website = "Não encontrado"
        instagram = "Não encontrado"

        # ------------------------
        # TELEFONE
        # ------------------------
        try:
            tel_element = page.locator("[data-item-id*='phone']")
            if await tel_element.count() > 0:
                telefone = await tel_element.get_attribute("aria-label")
                telefone = telefone.replace("Telefone:", "").strip()
        except:
            pass

        # Fallback pelo texto do painel
        if telefone == "Não encontrado":
            try:
                painel = page.locator("div[role='main']")
                texto_painel = await painel.inner_text()
                telefones = re.findall(r'\(?\d{2}\)?\s?\d{4,5}-?\d{4}', texto_painel)
                telefone = telefones[0] if telefones else "Não encontrado"
            except:
                pass

        # ------------------------
        # LINKS GERAIS
        # ------------------------
        # Domínios que queremos ignorar (são links do próprio Google)
        dominios_ignorar = [
            "google.com",
            "maps.google",
            "goo.gl",
            "policies.google",
            "support.google",
            "accounts.google"
        ]

        # Domínios de redes sociais conhecidas
        redes_sociais = {
            "instagram.com": "instagram",
            "facebook.com": "facebook",
            "twitter.com": "twitter",
            "x.com": "twitter",
            "tiktok.com": "tiktok",
            "youtube.com": "youtube",
            "linkedin.com": "linkedin",
            "whatsapp.com": "whatsapp"
        }

        links_encontrados = {
            "website": "Não encontrado",
            "instagram": "Não encontrado",
            "facebook": "Não encontrado",
            "twitter": "Não encontrado",
            "tiktok": "Não encontrado",
            "youtube": "Não encontrado",
            "linkedin": "Não encontrado",
            "whatsapp": "Não encontrado"
        }

        try:
            todos_links = page.locator("div[role='main'] a")
            quantidade = await todos_links.count()

            # Usamos um set para evitar duplicatas
            hrefs_vistos = set()

            for j in range(quantidade):
                href = await todos_links.nth(j).get_attribute("href")

                # Ignora links vazios ou já vistos
                if not href or href in hrefs_vistos:
                    continue

                hrefs_vistos.add(href)

                # Ignora links do próprio Google
                if any(dominio in href for dominio in dominios_ignorar):
                    continue

                # Verifica se é uma rede social conhecida
                eh_rede_social = False
                for dominio, nome_rede in redes_sociais.items():
                    if dominio in href:
                        # Só salva se ainda não encontrou essa rede social
                        if links_encontrados[nome_rede] == "Não encontrado":
                            links_encontrados[nome_rede] = href
                        eh_rede_social = True
                        break

                # Se não é rede social e não é do Google, é um website
                if not eh_rede_social:
                    if links_encontrados["website"] == "Não encontrado":
                        links_encontrados["website"] = href

        except:
            pass

        return {
            "telefone": telefone,
            **links_encontrados  # Desempacota todos os links
        }

    except Exception as e:
        return {
            "telefone": "Não encontrado",
            "website": "Não encontrado",
            "instagram": "Não encontrado",
            "facebook": "Não encontrado",
            "twitter": "Não encontrado",
            "tiktok": "Não encontrado",
            "youtube": "Não encontrado",
            "linkedin": "Não encontrado",
            "whatsapp": "Não encontrado"
        }
# ------------------------
# SCRAPER PRINCIPAL
# ------------------------
async def main(query):
    query_formatada = query.strip().replace(" ", "+")
    url = f"https://www.google.com/maps/search/{query_formatada}"

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=["--no-sandbox", "--disable-setuid-sandbox"]
        )
        page = await browser.new_page()

        await page.set_extra_http_headers({
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept-Language": "pt-BR,pt;q=0.9"
        })

        await page.goto(url, wait_until="domcontentloaded") # Mudança aqui!
        await asyncio.sleep(2) # Pequena pausa no lugar do networkidle

        # ------------------------
        # ACEITA OS COOKIES
        # ------------------------
        try:
            aceitar = page.locator("button:has-text('Aceitar tudo')")
            count = await aceitar.count()
            print(f"Botões encontrados: {count}")

            if count > 0:
                await aceitar.first.click()
                print("✅ Cookies aceitos!")
                await asyncio.sleep(1.5) # Espera o Maps carregar após aceitar

        except Exception as e:
            print(f"Erro ao aceitar cookies: {e}")

        # Espera o feed aparecer
        await page.wait_for_selector("div[role='feed']", timeout=15000)

        # ------------------------
        # SCROLL
        # ------------------------
        feed = page.locator("div[role='feed']")
        timer = time.time()

        for _ in range(20):
            await feed.evaluate("el => el.scrollTop = el.scrollHeight")
            await asyncio.sleep(1)

            places = page.locator("div.Nv2PK")
            count = await places.count()
            print(f"Resultados encontrados: {count}")

            if count >= 50:
                break

            if time.time() - timer > 30:
                print("⏱️ Tempo limite atingido")
                break

        # ------------------------
        # EXTRAÇÃO
        # ------------------------
        print("\nIniciando extração dos dados...\n")

        dados = []
        places = page.locator("div.Nv2PK")
        count = await places.count()

        for i in range(count):
            try:
                card = page.locator("div.Nv2PK").nth(i)
                texto_completo = await card.inner_text()
                texto_completo = limpar_texto(texto_completo)
                nome = texto_completo.split("\n")[0]

                print(f"[{i+1}/{count}] Extraindo: {nome}...")

                detalhes = await extrair_detalhes(page, card)

                dados.append({
                    "nome": nome,
                    **detalhes
                })

                print(f"  ✅ Telefone: {detalhes['telefone']}")
                print(f"  🌐 Website: {detalhes['website']}")
                print(f"  📸 Instagram: {detalhes['instagram']}")
                print(f"  👍 Facebook: {detalhes['facebook']}")
                print(f"  🎵 TikTok: {detalhes['tiktok']}")

                await page.keyboard.press("Escape")
                await asyncio.sleep(0.5)

            except Exception as e:
                print(f"  ❌ Erro no card {i}: {e}")
                continue

        await browser.close()
        return dados

# 3 - Query
Rode a célula abaixo, irá abrir uma caixinha perguntando o que gostaria de rodar no maps.


In [3]:
# ------------------------
# EXECUÇÃO
# ------------------------
query = input("Digite o que gostaria de pesquisar no google maps: ")
resultados = await main(query)
print(f"\n✅ Total extraído: {len(resultados)} lugares")

# Salva em CSV
import pandas as pd
df = pd.DataFrame(resultados)
df.to_csv("resultados_maps.csv", index=False, encoding="utf-8-sig")
print("📁 Salvo em: resultados_maps.csv")

from google.colab import files
files.download("resultados_maps.csv")

Digite o que gostaria de pesquisar no google maps: Restaurantes em cotia
Botões encontrados: 0
Resultados encontrados: 12
Resultados encontrados: 18
Resultados encontrados: 20
Resultados encontrados: 26
Resultados encontrados: 32
Resultados encontrados: 38
Resultados encontrados: 40
Resultados encontrados: 40
Resultados encontrados: 40
Resultados encontrados: 46
Resultados encontrados: 52

Iniciando extração dos dados...

[1/52] Extraindo: Restaurante Osaka - Unidade Cotia...
  ✅ Telefone: Não encontrado
  🌐 Website: Não encontrado
  📸 Instagram: Não encontrado
  👍 Facebook: Não encontrado
  🎵 TikTok: Não encontrado
[2/52] Extraindo: A Quinta do Bacalhau...
  ✅ Telefone: +55 11 4777-0546
  🌐 Website: tel:+551147770546
  📸 Instagram: https://instagram.com/osakacotia?igshid=18mj7zs0ukqz7
  👍 Facebook: Não encontrado
  🎵 TikTok: Não encontrado
[3/52] Extraindo: Don Camillo Ristorante Italiano...
  ✅ Telefone: +55 11 4777-0546
  🌐 Website: tel:+551147770546
  📸 Instagram: https://instagram

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>